In [32]:
!pip install pymupdf pandas tiktoken

In [1]:
# ──────────────────────────────────────────────
# STEP 1 – Imports
# ──────────────────────────────────────────────

import os
import re
import json
import sqlite3
import traceback
from pathlib import Path
from statistics import mean
from collections import Counter, defaultdict

import fitz  # PyMuPDF
import pandas as pd
import tiktoken

In [34]:
# ──────────────────────────────────────────────
# STEP 2A – Colab 
# ──────────────────────────────────────────────

from google.colab import files

uploaded = files.upload()

pdf_files = [Path(name) for name in uploaded.keys()]

print(f"Uploaded PDF files found: {len(pdf_files)}")
for pdf in pdf_files:
    print("-", pdf.name)

ModuleNotFoundError: No module named 'google.colab'

In [12]:
# ──────────────────────────────────────────────
# STEP 2B – Local testing without Colab
# ──────────────────────────────────────────────

CORPUS_DIR = Path("..") / "legal_corpus"
pdf_files = sorted(CORPUS_DIR.glob("*.pdf"))

print(f"Corpus directory: {CORPUS_DIR.resolve()}")
print(f"PDF files found: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", pdf.name)

Corpus directory: /Users/claycasani/PyCharmProjects/BTE440/legal-ai-capstone/legal_corpus
PDF files found: 20
- Big River Contract_Redacted.pdf
- Coconut Grove Commercial Contract_Redacted.pdf
- Executed - Mutual Confidentiality - NDA- Nili-Marine - Redacted.pdf
- Small River Contract_Redacted.pdf
- consulting-agreement_1099-contractor-template.pdf
- consulting-agreement_generic.pdf
- consulting-agreement_osrp_2021-04.pdf
- executive-employment-agreement_global-technologies_2024-11-22.pdf
- independent-contractor-agreement_long-form.pdf
- lease-agreement_signed.pdf
- master-service-agreement_sample.pdf
- nda-non-solicitation_edc-ventures.pdf
- nda_dhs-form-11000-6.pdf
- nda_mutual-template.pdf
- professional-services-agreement_cornell-university.pdf
- saas-agreement_cooley-acc.pdf
- saas-agreement_onestream.pdf
- saas-agreement_online_v1.0.pdf
- stock-subscription-agreement_global-technologies_series-p.pdf
- yacht-charter-bareboat-agreement_salvaje_2022-11-12.pdf


In [3]:
# For local testing without Colab, you can use this code to read a file path from the user:

source_path = input("Paste the full file path: ").strip()
file_name = source_path.split("/")[-1]

with open(source_path, "rb") as f:
    file_bytes = f.read()

uploaded = {file_name: file_bytes}

with open(file_name, "wb") as f:
    f.write(file_bytes)


KeyboardInterrupt: Interrupted by user

In [13]:
# ──────────────────────────────────────────────
# STEP 3 – Token Counter
# ──────────────────────────────────────────────

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(text))

In [14]:
# ──────────────────────────────────────────────
# STEP 3 – Parser
# ──────────────────────────────────────────────

import fitz  # PyMuPDF
import re
from collections import Counter

# ---------- CLEANING ----------
def clean_extracted_text(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ---------- HEADER/FOOTER REMOVAL ----------
def remove_repeated_lines(text, min_repeats=3):
    lines = [line.strip() for line in text.splitlines()]
    counts = Counter(line for line in lines if line)
    repeated = {line for line, c in counts.items() if c >= min_repeats}

    cleaned_lines = []
    for line in lines:
        if line in repeated and len(line) < 120:
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

# ---------- MAIN PARSER ----------
def parse_pdf(file_path):
    doc = fitz.open(str(file_path))
    pages = []

    for page in doc:
        page_text = page.get_text("text")
        pages.append(page_text)

    full_text = "\n\n".join(pages)
    full_text = clean_extracted_text(full_text)
    full_text = remove_repeated_lines(full_text)
    return full_text

In [15]:
# ──────────────────────────────────────────────
# STEP 5 – Clause-Aware Chunking (Improved)
# ──────────────────────────────────────────────

import re

CLAUSE_HEADER_PATTERN = re.compile(
    r'(?im)^(section\s+\d+(\.\d+)*|article\s+[ivx\d]+|'
    r'\d+(\.\d+)*\s+[A-Z][^\n]*|[A-Z][A-Z \-/]{4,})$'
)

def split_into_sections(text):
    lines = text.splitlines()
    sections = []
    current = []

    for line in lines:
        stripped = line.strip()
        if CLAUSE_HEADER_PATTERN.match(stripped):
            if current:
                sections.append("\n".join(current).strip())
                current = []
        current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return [s for s in sections if s.strip()]


def build_chunk_from_words(words, chunk_index, char_cursor, min_tokens=800, max_tokens=1000):
    chunk_text = " ".join(words).strip()
    token_count = count_tokens(chunk_text)

    return {
        "text": chunk_text,
        "chunk_index": chunk_index,
        "token_count": token_count,
        "start_char": char_cursor,
        "end_char": char_cursor + len(chunk_text),
        "out_of_range": not (min_tokens <= token_count <= max_tokens)
    }


def trim_to_max_tokens(words, max_tokens):
    words = words[:]
    while words and count_tokens(" ".join(words)) > max_tokens:
        words.pop()
    return words


def get_overlap_words(words, target_overlap_tokens=175):
    overlap_words = []
    for word in reversed(words):
        overlap_words.insert(0, word)
        if count_tokens(" ".join(overlap_words)) >= target_overlap_tokens:
            break
    return overlap_words


def merge_small_chunks(chunks, min_tokens=800, max_tokens=1000):
    """
    Merge undersized chunks with the following chunk when possible.
    """
    if not chunks:
        return chunks

    merged = []
    i = 0

    while i < len(chunks):
        current = chunks[i]

        if current["token_count"] >= min_tokens or i == len(chunks) - 1:
            merged.append(current)
            i += 1
            continue

        nxt = chunks[i + 1]
        combined_text = (current["text"] + " " + nxt["text"]).strip()
        combined_tokens = count_tokens(combined_text)

        if combined_tokens <= max_tokens:
            merged_chunk = {
                "text": combined_text,
                "chunk_index": current["chunk_index"],
                "token_count": combined_tokens,
                "start_char": current["start_char"],
                "end_char": current["start_char"] + len(combined_text),
                "out_of_range": not (min_tokens <= combined_tokens <= max_tokens)
            }
            merged.append(merged_chunk)
            i += 2
        else:
            merged.append(current)
            i += 1

    for idx, chunk in enumerate(merged):
        chunk["chunk_index"] = idx

    return merged


def split_large_section(section_words, chunk_size=900, overlap=175, min_tokens=800, max_tokens=1000,
                        start_chunk_index=0, start_char_cursor=0):
    chunks = []
    chunk_index = start_chunk_index
    char_cursor = start_char_cursor
    remaining = section_words[:]

    while remaining:
        candidate = []
        i = 0

        while i < len(remaining):
            trial = candidate + [remaining[i]]
            if count_tokens(" ".join(trial)) <= chunk_size:
                candidate = trial
                i += 1
            else:
                break

        if not candidate and remaining:
            candidate = [remaining[0]]
            i = 1

        candidate = trim_to_max_tokens(candidate, max_tokens)

        chunk = build_chunk_from_words(candidate, chunk_index, char_cursor, min_tokens, max_tokens)
        chunks.append(chunk)

        chunk_index += 1
        char_cursor = chunk["end_char"]

        overlap_words = get_overlap_words(candidate, overlap)
        remaining = overlap_words + remaining[i:]

        # prevent infinite loop
        if len(remaining) == len(overlap_words):
            break

    return chunks, chunk_index, char_cursor


def chunk_document(text, chunk_size=900, overlap=175, min_tokens=800, max_tokens=1000):
    sections = split_into_sections(text)

    chunks = []
    current_words = []
    chunk_index = 0
    char_cursor = 0

    for section in sections:
        section_words = section.split()
        section_text = " ".join(section_words)

        if count_tokens(section_text) > max_tokens:
            if current_words:
                current_words = trim_to_max_tokens(current_words, max_tokens)
                chunk = build_chunk_from_words(current_words, chunk_index, char_cursor, min_tokens, max_tokens)
                chunks.append(chunk)

                chunk_index += 1
                char_cursor = chunk["end_char"]
                current_words = get_overlap_words(current_words, overlap)

            large_chunks, chunk_index, char_cursor = split_large_section(
                section_words,
                chunk_size=chunk_size,
                overlap=overlap,
                min_tokens=min_tokens,
                max_tokens=max_tokens,
                start_chunk_index=chunk_index,
                start_char_cursor=char_cursor
            )
            chunks.extend(large_chunks)
            current_words = []
            continue

        trial_words = current_words + section_words
        trial_text = " ".join(trial_words)
        trial_tokens = count_tokens(trial_text)

        if trial_tokens <= chunk_size:
            current_words = trial_words
        else:
            if current_words:
                current_words = trim_to_max_tokens(current_words, max_tokens)
                chunk = build_chunk_from_words(current_words, chunk_index, char_cursor, min_tokens, max_tokens)
                chunks.append(chunk)

                chunk_index += 1
                char_cursor = chunk["end_char"]

                overlap_words = get_overlap_words(current_words, overlap)
                current_words = overlap_words + section_words
            else:
                temp_chunks, chunk_index, char_cursor = split_large_section(
                    section_words,
                    chunk_size=chunk_size,
                    overlap=overlap,
                    min_tokens=min_tokens,
                    max_tokens=max_tokens,
                    start_chunk_index=chunk_index,
                    start_char_cursor=char_cursor
                )
                chunks.extend(temp_chunks)
                current_words = []

    if current_words:
        current_words = trim_to_max_tokens(current_words, max_tokens)
        chunk = build_chunk_from_words(current_words, chunk_index, char_cursor, min_tokens, max_tokens)
        chunks.append(chunk)

    chunks = merge_small_chunks(chunks, min_tokens=min_tokens, max_tokens=max_tokens)
    return chunks

In [16]:
# ──────────────────────────────────────────────
# STEP 6 – Clause Metadata Tagging
# ──────────────────────────────────────────────

CLAUSE_PATTERNS = {
    "Payment / financial terms": [
        r"\bpayment\b", r"\bpayments\b", r"\bpay\b", r"\bfee\b", r"\bfees\b",
        r"\bcompensation\b", r"\binvoice\b", r"\binvoices\b", r"\bbilling\b",
        r"\brate\b", r"\brates\b", r"\bprice\b", r"\bpricing\b",
        r"\breimbursement\b", r"\breimburse\b", r"\bexpense\b", r"\bexpenses\b",
        r"\bdue\b", r"\bdeposit\b"
    ],
    "Termination / cancellation": [
        r"\bterminate\b", r"\btermination\b", r"\bcancel\b", r"\bcancellation\b",
        r"\bexpired\b", r"\bexpiration\b", r"\bnotice\b", r"\bbreach\b"
    ],
    "Liability / indemnification": [
        r"\bliability\b", r"\bliable\b", r"\bindemnify\b", r"\bindemnification\b",
        r"\bhold harmless\b", r"\bdamages\b", r"\blosses\b", r"\bclaims\b"
    ],
    "IP ownership / licensing": [
        r"\bintellectual property\b", r"\bownership\b", r"\bown\b",
        r"\blicense\b", r"\blicensing\b", r"\bcopyright\b",
        r"\btrademark\b", r"\bpatent\b", r"\bwork product\b"
    ],
    "Confidentiality / non-disclosure": [
        r"\bconfidential\b", r"\bconfidentiality\b", r"\bnon[- ]disclosure\b",
        r"\bnda\b", r"\bproprietary information\b", r"\bdisclosure\b"
    ],
    "Governing law / jurisdiction": [
        r"\bgoverning law\b", r"\bjurisdiction\b", r"\bvenue\b",
        r"\blaws of\b", r"\bcourts of\b"
    ],
    "Dispute resolution / arbitration": [
        r"\bdispute\b", r"\bdisputes\b", r"\barbitration\b",
        r"\bmediate\b", r"\bmediation\b", r"\bresolution\b"
    ],
    "Warranties / representations": [
        r"\bwarranty\b", r"\bwarranties\b", r"\brepresentation\b",
        r"\brepresentations\b", r"\bguarantee\b", r"\bguarantees\b",
        r"\bas is\b"
    ]
}

def tag_clause_metadata(chunk_text):
    tags = []
    text = chunk_text.lower()

    for clause_type, patterns in CLAUSE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                tags.append(clause_type)
                break

    return tags

In [17]:
# ──────────────────────────────────────────────
# STEP 7 – Full Corpus Processing Pipeline
# ──────────────────────────────────────────────

corpus_results = []
pipeline_errors = []
parsed_results = {}
all_chunks = {}
tagged_chunks = []

for pdf_path in pdf_files:
    file_name = pdf_path.name
    print(f"\nProcessing: {file_name}")

    try:
        # 1. Parse
        parsed_text = parse_pdf(pdf_path)
        parsed_results[file_name] = parsed_text

        # 2. Chunk
        chunks = chunk_document(
            parsed_text,
            chunk_size=900,
            overlap=175,
            min_tokens=800,
            max_tokens=1000
        )
        all_chunks[file_name] = chunks

        # 3. Tag
        token_counts = []
        out_of_range_count = 0

        for chunk in chunks:
            chunk_id = f"{file_name}::chunk_{chunk['chunk_index']}"

            chunk_obj = {
                "chunk_id": chunk_id,
                "chunk_text": chunk["text"],
                "chunk_index": chunk["chunk_index"],
                "source_document": file_name,
                "clause_tags": tag_clause_metadata(chunk["text"]),
                "token_count": chunk["token_count"],
                "start_char": chunk["start_char"],
                "end_char": chunk["end_char"],
                "out_of_range": chunk["out_of_range"]
            }

            tagged_chunks.append(chunk_obj)
            token_counts.append(chunk["token_count"])

            if chunk["out_of_range"]:
                out_of_range_count += 1

        corpus_results.append({
            "source_document": file_name,
            "parse_success": True,
            "character_count": len(parsed_text),
            "num_chunks": len(chunks),
            "avg_chunk_tokens": round(mean(token_counts), 2) if token_counts else 0,
            "min_chunk_tokens": min(token_counts) if token_counts else 0,
            "max_chunk_tokens": max(token_counts) if token_counts else 0,
            "out_of_range_chunks": out_of_range_count,
            "error_message": ""
        })

        print(f"Finished: {file_name}")
        print(f"  Parsed chars: {len(parsed_text)}")
        print(f"  Chunks: {len(chunks)}")
        print(f"  Avg chunk tokens: {round(mean(token_counts), 2) if token_counts else 0}")
        print(f"  Out-of-range chunks: {out_of_range_count}")

    except Exception as e:
        pipeline_errors.append({
            "source_document": file_name,
            "error_message": str(e),
            "traceback": traceback.format_exc()
        })

        corpus_results.append({
            "source_document": file_name,
            "parse_success": False,
            "character_count": 0,
            "num_chunks": 0,
            "avg_chunk_tokens": 0,
            "min_chunk_tokens": 0,
            "max_chunk_tokens": 0,
            "out_of_range_chunks": 0,
            "error_message": str(e)
        })

        print(f"FAILED: {file_name}")
        print(str(e))

print("\nFull corpus processing complete.")
print(f"Documents found: {len(pdf_files)}")
print(f"Documents processed: {len(corpus_results)}")
print(f"Successful: {sum(1 for r in corpus_results if r['parse_success'])}")
print(f"Failed: {sum(1 for r in corpus_results if not r['parse_success'])}")
print(f"Total tagged chunks: {len(tagged_chunks)}")


Processing: Big River Contract_Redacted.pdf
Finished: Big River Contract_Redacted.pdf
  Parsed chars: 36980
  Chunks: 15
  Avg chunk tokens: 737.33
  Out-of-range chunks: 9

Processing: Coconut Grove Commercial Contract_Redacted.pdf
Finished: Coconut Grove Commercial Contract_Redacted.pdf
  Parsed chars: 38298
  Chunks: 14
  Avg chunk tokens: 780.29
  Out-of-range chunks: 5

Processing: Executed - Mutual Confidentiality - NDA- Nili-Marine - Redacted.pdf
Finished: Executed - Mutual Confidentiality - NDA- Nili-Marine - Redacted.pdf
  Parsed chars: 16308
  Chunks: 5
  Avg chunk tokens: 758
  Out-of-range chunks: 2

Processing: Small River Contract_Redacted.pdf
Finished: Small River Contract_Redacted.pdf
  Parsed chars: 36945
  Chunks: 14
  Avg chunk tokens: 749.14
  Out-of-range chunks: 8

Processing: consulting-agreement_1099-contractor-template.pdf
Finished: consulting-agreement_1099-contractor-template.pdf
  Parsed chars: 13127
  Chunks: 4
  Avg chunk tokens: 755.25
  Out-of-range chu

In [18]:
# ──────────────────────────────────────────────
# OPTIONAL – Parsed Text Summary
# ──────────────────────────────────────────────

parsed_results = {}

for file_name, text in parsed_results.items():
    print("=" * 80)
    print(f"File: {file_name}")
    print(f"Character count: {len(text)}")
    print(f"Word count: {len(text.split())}")
print("=" * 80)

for file_name, text in parsed_results.items():
    print(f"\nFile: {file_name}")
    print(f"Character count: {len(text)}")
    print(f"Word count: {len(text.split())}")
    print("-"*50)

In [19]:
# ──────────────────────────────────────────────
# OPTIONAL – Save Parsed Raw Text Outputs
# ──────────────────────────────────────────────

from pathlib import Path


OUTPUT_DIR = Path("milestone1_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

parsed_dir = Path(OUTPUT_DIR) / "parsed_texts"
parsed_dir.mkdir(exist_ok=True)

for file_name, text in parsed_results.items():
    output_name = parsed_dir / f"{Path(file_name).stem}.txt"

    with open(output_name, "w", encoding="utf-8") as f:
        f.write(text)

print(f"Saved parsed text files to: {parsed_dir}")

Output directory: /Users/claycasani/PyCharmProjects/BTE440/legal-ai-capstone/notebooks/milestone1_outputs
Saved parsed text files to: milestone1_outputs/parsed_texts


In [20]:
# ──────────────────────────────────────────────
# STEP 9 – Save Tagged Chunks to JSONL
# ──────────────────────────────────────────────

jsonl_path = OUTPUT_DIR / "tagged_chunks.jsonl"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for chunk in tagged_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Saved JSONL to: {jsonl_path}")
print(f"Total chunk records written: {len(tagged_chunks)}")

Saved JSONL to: milestone1_outputs/tagged_chunks.jsonl
Total chunk records written: 308


In [21]:
# ──────────────────────────────────────────────
# STEP 10 – Save Tagged Chunks to SQLite
# ──────────────────────────────────────────────

sqlite_path = OUTPUT_DIR / "tagged_chunks.db"

conn = sqlite3.connect(sqlite_path)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS tagged_chunks")

cur.execute("""
CREATE TABLE tagged_chunks (
    chunk_id TEXT PRIMARY KEY,
    source_document TEXT,
    chunk_index INTEGER,
    chunk_text TEXT,
    clause_tags TEXT,
    token_count INTEGER,
    start_char INTEGER,
    end_char INTEGER,
    out_of_range INTEGER
)
""")

for chunk in tagged_chunks:
    cur.execute("""
        INSERT INTO tagged_chunks (
            chunk_id,
            source_document,
            chunk_index,
            chunk_text,
            clause_tags,
            token_count,
            start_char,
            end_char,
            out_of_range
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        chunk["chunk_id"],
        chunk["source_document"],
        chunk["chunk_index"],
        chunk["chunk_text"],
        json.dumps(chunk["clause_tags"]),
        chunk["token_count"],
        chunk["start_char"],
        chunk["end_char"],
        int(chunk["out_of_range"])
    ))

conn.commit()
conn.close()

print(f"Saved SQLite DB to: {sqlite_path}")

Saved SQLite DB to: milestone1_outputs/tagged_chunks.db


In [22]:
# ──────────────────────────────────────────────
# STEP 11 – Save Processing Logs
# ──────────────────────────────────────────────

logs_df = pd.DataFrame(corpus_results)

logs_csv_path = OUTPUT_DIR / "processing_logs.csv"
logs_json_path = OUTPUT_DIR / "processing_logs.json"

logs_df.to_csv(logs_csv_path, index=False)
logs_df.to_json(logs_json_path, orient="records", indent=2)

print("Processing logs saved.")
print("CSV log:", logs_csv_path)
print("JSON log:", logs_json_path)

display(logs_df)

Processing logs saved.
CSV log: milestone1_outputs/processing_logs.csv
JSON log: milestone1_outputs/processing_logs.json


,source_document,parse_success,character_count,num_chunks,avg_chunk_tokens,min_chunk_tokens,max_chunk_tokens,out_of_range_chunks,error_message
0,Big River Contract_Redacted.pdf,True,36980,15,737.33,545,974,9,
1,Coconut Grove Commercial Contract_Redacted.pdf,True,38298,14,780.29,383,1000,5,
2,Executed - Mutual Confidentiality - NDA- Nili-...,True,16308,5,758.00,483,899,2,
3,Small River Contract_Redacted.pdf,True,36945,14,749.14,448,965,8,
4,consulting-agreement_1099-contractor-template.pdf,True,13127,4,755.25,583,888,3,
5,consulting-agreement_generic.pdf,True,23057,7,758.71,491,896,2,
6,consulting-agreement_osrp_2021-04.pdf,True,14124,6,614.50,288,857,5,
7,executive-employment-agreement_global-technolo...,True,13915,5,674.00,207,1000,3,
8,independent-contractor-agreement_long-form.pdf,True,37681,11,800.00,613,880,4,
9,lease-agreement_signed.pdf,True,268567,106,791.57,314,1000,40,


In [23]:
# ──────────────────────────────────────────────
# STEP 12 – Save Pipeline Errors
# ──────────────────────────────────────────────

error_log_path = OUTPUT_DIR / "pipeline_errors.json"

with open(error_log_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_errors, f, indent=2, ensure_ascii=False)

print(f"Saved error log to: {error_log_path}")
print(f"Total errors recorded: {len(pipeline_errors)}")

Saved error log to: milestone1_outputs/pipeline_errors.json
Total errors recorded: 0


In [24]:
# ──────────────────────────────────────────────
# STEP 13 – QA: Chunk Size Distribution
# ──────────────────────────────────────────────

chunk_df = pd.DataFrame(tagged_chunks)

if len(chunk_df) == 0:
    print("No tagged chunks found.")
else:
    in_range_mask = chunk_df["token_count"].between(800, 1000)
    pct_in_range = round(in_range_mask.mean() * 100, 2)

    print("Chunk Size QA")
    print(f"Total chunks: {len(chunk_df)}")
    print(f"Min token count: {chunk_df['token_count'].min()}")
    print(f"Max token count: {chunk_df['token_count'].max()}")
    print(f"Average token count: {round(chunk_df['token_count'].mean(), 2)}")
    print(f"Chunks in 800–1000 range: {in_range_mask.sum()} / {len(chunk_df)}")
    print(f"Percent in range: {pct_in_range}%")

    display(chunk_df[["source_document", "chunk_index", "token_count", "out_of_range"]].head(20))

Chunk Size QA
Total chunks: 308
Min token count: 207
Max token count: 1000
Average token count: 772.27
Chunks in 800–1000 range: 177 / 308
Percent in range: 57.47%


,source_document,chunk_index,token_count,out_of_range
0,Big River Contract_Redacted.pdf,0,808,False
1,Big River Contract_Redacted.pdf,1,830,False
2,Big River Contract_Redacted.pdf,2,760,True
3,Big River Contract_Redacted.pdf,3,719,True
4,Big River Contract_Redacted.pdf,4,974,False
5,Big River Contract_Redacted.pdf,5,879,False
6,Big River Contract_Redacted.pdf,6,719,True
7,Big River Contract_Redacted.pdf,7,545,True
8,Big River Contract_Redacted.pdf,8,547,True
9,Big River Contract_Redacted.pdf,9,902,False


In [65]:
# ──────────────────────────────────────────────
# STEP 14 – QA: Out-of-Range Chunks
# ──────────────────────────────────────────────

out_of_range_df = chunk_df[~chunk_df["token_count"].between(800, 1000)].copy()

print(f"Out-of-range chunks: {len(out_of_range_df)}")
display(out_of_range_df[["source_document", "chunk_index", "token_count", "chunk_text"]].head(20))

Out-of-range chunks: 107


,source_document,chunk_index,token_count,chunk_text
0,Attachment 2 - Sample Master Service Agreement...,0,596,** This is only Mercy Corps’ standard template...
1,Attachment 2 - Sample Master Service Agreement...,1,662,TO. b. Contractor will perform all Services th...
2,Attachment 2 - Sample Master Service Agreement...,2,761,"Terms, an itemization of the specified increme..."
3,Attachment 2 - Sample Master Service Agreement...,3,622,Contractor has the requisite skills to perform...
8,Attachment 2 - Sample Master Service Agreement...,8,356,EXHIBIT A FORM TASK ORDER - FIXED PRICE Task O...
9,CONSULTING-AGREEMENT-template-for-1099-ee.pdf,0,770,1 CONSULTING AGREEMENT THIS CONSULTING AGREEME...
11,CONSULTING-AGREEMENT-template-for-1099-ee.pdf,2,780,constitutes the exclusive property of the Comp...
12,CONSULTING-AGREEMENT-template-for-1099-ee.pdf,3,583,according to the rules of the American Arbitra...
13,Cooley SaaS Agreement ACC Form.pdf,0,658,The below form is made available for general i...
18,Cooley SaaS Agreement ACC Form.pdf,5,695,"“SOW”). Each SOW will describe the fees, costs..."


In [25]:
# ──────────────────────────────────────────────
# STEP 15 – QA: Overlap Check
# ──────────────────────────────────────────────

def compute_word_overlap(text_a, text_b, max_check=250):
    words_a = text_a.split()
    words_b = text_b.split()

    tail_a = words_a[-max_check:]
    head_b = words_b[:max_check]

    best = 0
    max_possible = min(len(tail_a), len(head_b))

    for k in range(1, max_possible + 1):
        if tail_a[-k:] == head_b[:k]:
            best = k
    return best

overlap_rows = []

for file_name, chunks in all_chunks.items():
    for i in range(len(chunks) - 1):
        overlap_count = compute_word_overlap(chunks[i]["text"], chunks[i + 1]["text"], max_check=250)
        overlap_rows.append({
            "source_document": file_name,
            "chunk_index_a": chunks[i]["chunk_index"],
            "chunk_index_b": chunks[i + 1]["chunk_index"],
            "measured_overlap_tokens": overlap_count
        })

overlap_df = pd.DataFrame(overlap_rows)

if len(overlap_df) == 0:
    print("No overlap data available.")
else:
    print("Overlap QA")
    print(f"Average measured overlap: {round(overlap_df['measured_overlap_tokens'].mean(), 2)}")
    print(f"Min overlap: {overlap_df['measured_overlap_tokens'].min()}")
    print(f"Max overlap: {overlap_df['measured_overlap_tokens'].max()}")

    display(overlap_df.head(20))

Overlap QA
Average measured overlap: 123.08
Min overlap: 0
Max overlap: 157


,source_document,chunk_index_a,chunk_index_b,measured_overlap_tokens
0,Big River Contract_Redacted.pdf,0,1,137
1,Big River Contract_Redacted.pdf,1,2,148
2,Big River Contract_Redacted.pdf,2,3,143
3,Big River Contract_Redacted.pdf,3,4,135
4,Big River Contract_Redacted.pdf,4,5,114
5,Big River Contract_Redacted.pdf,5,6,129
6,Big River Contract_Redacted.pdf,6,7,123
7,Big River Contract_Redacted.pdf,7,8,140
8,Big River Contract_Redacted.pdf,8,9,112
9,Big River Contract_Redacted.pdf,9,10,88


In [26]:
# ──────────────────────────────────────────────
# STEP 16 – QA: Manual Spot Check Helper
# ──────────────────────────────────────────────

def preview_doc_chunks(file_name, preview_chars=1200):
    chunks = [c for c in tagged_chunks if c["source_document"] == file_name]
    if not chunks:
        print(f"No chunks found for {file_name}")
        return

    positions = [0, len(chunks)//2, len(chunks)-1]
    seen = set()

    print(f"\nManual spot check for: {file_name}")
    print(f"Total chunks: {len(chunks)}\n")

    for pos in positions:
        if pos in seen:
            continue
        seen.add(pos)
        c = chunks[pos]
        print("=" * 100)
        print(f"Chunk index: {c['chunk_index']}")
        print(f"Token count: {c['token_count']}")
        print(f"Clause tags: {c['clause_tags']}")
        print(f"Start char: {c['start_char']} | End char: {c['end_char']}")
        print("-" * 100)
        print(c["chunk_text"][:preview_chars])
        print("\n")

In [27]:
# ──────────────────────────────────────────────
# STEP 17 – QA: Manual Tag Validation Sample
# ──────────────────────────────────────────────

tag_validation_sample = chunk_df.sample(min(15, len(chunk_df)), random_state=42)[
    ["source_document", "chunk_index", "clause_tags", "token_count", "chunk_text"]
].copy()

display(tag_validation_sample)

,source_document,chunk_index,clause_tags,token_count,chunk_text
288,saas-agreement_online_v1.0.pdf,26,"[Payment / financial terms, Termination / canc...",893,submit its data processing facilities for audi...
9,Big River Contract_Redacted.pdf,9,"[Payment / financial terms, Termination / canc...",902,".-""""''s essential to Closing, fs disrupted, de..."
57,consulting-agreement_generic.pdf,5,"[Payment / financial terms, Termination / canc...",887,"date of termination. Upon termination, 7 Consu..."
60,consulting-agreement_osrp_2021-04.pdf,1,"[Payment / financial terms, Termination / canc...",699,comply with all applicable laws and regulation...
25,Coconut Grove Commercial Contract_Redacted.pdf,10,[Payment / financial terms],764,purchasing what is left of the Property at the...
63,consulting-agreement_osrp_2021-04.pdf,4,"[Termination / cancellation, IP ownership / li...",857,the express prior written consent of IIT; or (...
92,lease-agreement_signed.pdf,11,"[Payment / financial terms, Termination / canc...",784,exercise due care for your own and others' saf...
184,lease-agreement_signed.pdf,103,"[Payment / financial terms, Termination / canc...",844,Any resident parked in the “City of Coral Gabl...
244,saas-agreement_cooley-acc.pdf,24,"[Payment / financial terms, Termination / canc...",607,Provider: A Party may from time to time change...
46,Small River Contract_Redacted.pdf,12,"[Payment / financial terms, Warranties / repre...",527,328 329 BUYER AND SELLER AGREE THAT THIS SAID ...


In [28]:
# ──────────────────────────────────────────────
# STEP 18 – QA: Parsing Failure Check
# ──────────────────────────────────────────────

failed_docs_df = logs_df[logs_df["parse_success"] == False].copy()

print(f"Number of failed documents: {len(failed_docs_df)}")
display(failed_docs_df)

Number of failed documents: 0


,source_document,parse_success,character_count,num_chunks,avg_chunk_tokens,min_chunk_tokens,max_chunk_tokens,out_of_range_chunks,error_message


In [29]:
# ──────────────────────────────────────────────
# STEP 19 – QA: No Major Data Loss Check
# ──────────────────────────────────────────────

data_loss_rows = []

for file_name, original_text in parsed_results.items():
    chunks = all_chunks.get(file_name, [])
    if not chunks:
        continue

    original_word_count = len(original_text.split())
    sum_chunk_words = sum(len(c["text"].split()) for c in chunks)

    approx_overlap_total = max(len(chunks) - 1, 0) * 175
    approx_unique_words_from_chunks = sum_chunk_words - approx_overlap_total

    ratio = round(approx_unique_words_from_chunks / original_word_count, 3) if original_word_count else 0

    data_loss_rows.append({
        "source_document": file_name,
        "original_word_count": original_word_count,
        "sum_chunk_words": sum_chunk_words,
        "approx_unique_words_from_chunks": approx_unique_words_from_chunks,
        "coverage_ratio": ratio
    })

data_loss_df = pd.DataFrame(data_loss_rows)

print("No-data-loss sanity check")
display(data_loss_df)

No-data-loss sanity check


""


In [30]:
# ──────────────────────────────────────────────
# STEP 20 – Final Milestone 1 Summary
# ──────────────────────────────────────────────

total_docs = len(corpus_results)
successful_docs = sum(1 for r in corpus_results if r["parse_success"])
failed_docs = total_docs - successful_docs
total_chunks = len(tagged_chunks)

if total_chunks > 0:
    pct_in_range = round((chunk_df["token_count"].between(800, 1000).mean()) * 100, 2)
else:
    pct_in_range = 0

print("MILESTONE 1 SUMMARY")
print("=" * 50)
print(f"Documents processed: {total_docs}")
print(f"Successful parses: {successful_docs}")
print(f"Failed parses: {failed_docs}")
print(f"Total tagged chunks: {total_chunks}")
print(f"Percent of chunks in 800–1000 range: {pct_in_range}%")
print(f"JSONL saved: {OUTPUT_DIR / 'tagged_chunks.jsonl'}")
print(f"SQLite saved: {OUTPUT_DIR / 'tagged_chunks.db'}")
print(f"Logs saved: {OUTPUT_DIR / 'processing_logs.csv'}")
print(f"Error log saved: {OUTPUT_DIR / 'pipeline_errors.json'}")

MILESTONE 1 SUMMARY
Documents processed: 20
Successful parses: 20
Failed parses: 0
Total tagged chunks: 308
Percent of chunks in 800–1000 range: 57.47%
JSONL saved: milestone1_outputs/tagged_chunks.jsonl
SQLite saved: milestone1_outputs/tagged_chunks.db
Logs saved: milestone1_outputs/processing_logs.csv
Error log saved: milestone1_outputs/pipeline_errors.json


In [73]:
# ──────────────────────────────────────────────
# DIAGNOSTIC 1 – classify out-of-range chunks
# ──────────────────────────────────────────────

import pandas as pd

chunk_df = pd.DataFrame(tagged_chunks)

def size_bucket(token_count):
    if token_count < 800:
        return "too_small"
    elif token_count > 1000:
        return "too_large"
    else:
        return "in_range"

chunk_df["size_bucket"] = chunk_df["token_count"].apply(size_bucket)

summary = (
    chunk_df.groupby(["source_document", "size_bucket"])
    .size()
    .unstack(fill_value=0)
)

# make sure all expected columns exist
for col in ["too_small", "too_large", "in_range"]:
    if col not in summary.columns:
        summary[col] = 0

summary = summary.reset_index()

display(
    summary.sort_values(by=["too_small", "too_large"], ascending=False)
)

size_bucket,source_document,in_range,too_small,too_large
15,signed_packet_file_lease_document_18725351_175...,66,40,0
2,Cooley SaaS Agreement ACC Form.pdf,13,12,0
14,prof-services-agrmt.pdf,8,9,0
7,Online-SaaS-Agreement-v1.0.pdf,21,8,0
3,EnvelopePDF.aspx.pdf,8,7,0
0,Attachment 2 - Sample Master Service Agreement...,4,5,0
4,Form-Consulting-Agreement-OSRP-04-2021_April-F...,1,5,0
5,Independent Contractor Agreement (Long Form)(2...,7,4,0
1,CONSULTING-AGREEMENT-template-for-1099-ee.pdf,1,3,0
6,Onestream-SaaS-Agreement.pdf,14,3,0


In [74]:
# ──────────────────────────────────────────────
# DIAGNOSTIC 2 – show most extreme chunks
# ──────────────────────────────────────────────

too_small_df = chunk_df[chunk_df["token_count"] < 800].copy()
too_large_df = chunk_df[chunk_df["token_count"] > 1000].copy()

print("Most undersized chunks:")
display(
    too_small_df.sort_values("token_count")
    [["source_document", "chunk_index", "token_count", "clause_tags"]]
    .head(20)
)

print("\nMost oversized chunks:")
display(
    too_large_df.sort_values("token_count", ascending=False)
    [["source_document", "chunk_index", "token_count", "clause_tags"]]
    .head(20)
)

Most undersized chunks:


,source_document,chunk_index,token_count,clause_tags
133,ex10-1.pdf,4,207,[Payment / financial terms]
153,prof-services-agrmt.pdf,16,270,"[Payment / financial terms, Liability / indemn..."
52,EnvelopePDF.aspx.pdf,14,284,[]
58,Form-Consulting-Agreement-OSRP-04-2021_April-F...,5,288,"[Termination / cancellation, Governing law / j..."
241,signed_packet_file_lease_document_18725351_175...,87,314,[]
189,signed_packet_file_lease_document_18725351_175...,35,333,"[Payment / financial terms, Termination / canc..."
252,signed_packet_file_lease_document_18725351_175...,98,334,"[Payment / financial terms, Termination / canc..."
233,signed_packet_file_lease_document_18725351_175...,79,347,"[Payment / financial terms, Termination / canc..."
8,Attachment 2 - Sample Master Service Agreement...,8,356,"[Payment / financial terms, Termination / canc..."
128,ex10-1-2.pdf,1,367,"[Payment / financial terms, IP ownership / lic..."



Most oversized chunks:


,source_document,chunk_index,token_count,clause_tags


In [75]:
# ──────────────────────────────────────────────
# DIAGNOSTIC 3 – inspect specific bad chunks
# ──────────────────────────────────────────────

def inspect_chunk(file_name, chunk_index, preview_chars=1800):
    matches = [
        c for c in tagged_chunks
        if c["source_document"] == file_name and c["chunk_index"] == chunk_index
    ]

    if not matches:
        print("Chunk not found.")
        return

    c = matches[0]
    print("=" * 100)
    print("File:", c["source_document"])
    print("Chunk index:", c["chunk_index"])
    print("Token count:", c["token_count"])
    print("Clause tags:", c["clause_tags"])
    print("Out of range:", c["out_of_range"])
    print("-" * 100)
    print(c["chunk_text"][:preview_chars])
    print("\n")

In [76]:
# ──────────────────────────────────────────────
# DIAGNOSTIC 4 – are final chunks causing most misses?
# ──────────────────────────────────────────────

final_chunk_rows = []

for file_name, chunks in all_chunks.items():
    if not chunks:
        continue
    last_chunk = chunks[-1]
    final_chunk_rows.append({
        "source_document": file_name,
        "chunk_index": last_chunk["chunk_index"],
        "token_count": last_chunk["token_count"],
        "out_of_range": last_chunk["out_of_range"]
    })

final_df = pd.DataFrame(final_chunk_rows)
display(final_df.sort_values("token_count"))

,source_document,chunk_index,token_count,out_of_range
11,ex10-1.pdf,4,207,True
14,prof-services-agrmt.pdf,16,270,True
3,EnvelopePDF.aspx.pdf,14,284,True
4,Form-Consulting-Agreement-OSRP-04-2021_April-F...,5,288,True
0,Attachment 2 - Sample Master Service Agreement...,8,356,True
10,ex10-1-2.pdf,1,367,True
8,consulting-agreement.pdf,6,491,True
6,Onestream-SaaS-Agreement.pdf,16,544,True
7,Online-SaaS-Agreement-v1.0.pdf,28,545,True
9,dhs_form_11000-6.pdf,3,582,True


In [77]:
# ──────────────────────────────────────────────
# DIAGNOSTIC 5 – section sizes before chunking
# ──────────────────────────────────────────────

def section_size_report(file_name):
    text = parsed_results[file_name]
    sections = split_into_sections(text)

    rows = []
    for i, section in enumerate(sections):
        rows.append({
            "section_index": i,
            "token_count": count_tokens(section),
            "preview": section[:150].replace("\n", " ")
        })

    df = pd.DataFrame(rows)
    print(f"Section report for: {file_name}")
    display(df.sort_values("token_count", ascending=False).head(20))